In [6]:
import cv2
import torch
import logging
from ultralytics import YOLO, solutions

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def process_video(model_path, video_path):
    """视频处理生成器，返回视频参数和帧数据"""
    model = YOLO(model_path)
    model.to(device)

    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        logging.error("Error 打开视频错误")
        return

    fps = capture.get(cv2.CAP_PROP_FPS)
    w = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    line_points = [(0, h//2), (w, h//2)]

    # 初始化ObjectCounter类
    counter = solutions.ObjectCounter(
        view_img=False,
        reg_pts=line_points,
        names=model.names,
        draw_tracks=True,
        line_thickness=1,
    )

    # 生成视频参数(帧数，帧宽，帧高)
    yield (fps, w, h)

    # 处理视频帧
    while capture.isOpened():
        success, frame = capture.read()
        if not success:
            logging.warning("视频帧读取结束")
            break

        tracks = model.track(frame, persist=True, show=False, device=device, classes=[2,3,5,7])
        processed_frame = counter.start_counting(frame, tracks)

        # 数据字典
        counts = {
            "in": counter.in_counts,
            "out": counter.out_counts,
            "class_wise": counter.class_wise_count.copy()
        }
        # 生成处理完的帧(processed_frame)和得到的信息(counts)
        yield (processed_frame, counts)

    # 释放资源
    capture.release()

def display_and_print(generator):
    """显示画面并打印计数信息"""
    try:
        # 获取视频基础参数
        fps, w, h = next(generator)
    except StopIteration:
        return

    while True:
        try:
            frame, counts = next(generator)
        except StopIteration:
            break

        # 显示画面
        cv2.imshow('output', frame)
        
        # 打印信息
        print(f"\n当前统计:")
        print(f"总进入(下到上): {counts['in']} | 总离开(上到下): {counts['out']}")
        for cls_name, cls_counts in counts['class_wise'].items():
            print(f"{cls_name} 进入: {cls_counts.get('IN',0)} | 离开: {cls_counts.get('OUT',0)}")

        # 退出控制，按q键
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()

In [4]:
video_gen = process_video("yolo11n.pt", "../kuruma.mp4")
display_and_print(video_gen)

Line Counter Initiated.

0: 384x640 9 cars, 2 buss, 14.0ms
Speed: 2.5ms preprocess, 14.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 0
bus 进入: 0 | 离开: 0
car 进入: 0 | 离开: 0

0: 384x640 9 cars, 2 buss, 12.7ms
Speed: 2.3ms preprocess, 12.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 2
bus 进入: 0 | 离开: 1
car 进入: 0 | 离开: 1

0: 384x640 9 cars, 2 buss, 11.9ms
Speed: 1.7ms preprocess, 11.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 2
bus 进入: 0 | 离开: 1
car 进入: 0 | 离开: 1

0: 384x640 9 cars, 2 buss, 11.1ms
Speed: 2.0ms preprocess, 11.1ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 2
bus 进入: 0 | 离开: 1
car 进入: 0 | 离开: 1

0: 384x640 9 cars, 2 buss, 13.1ms
Speed: 1.0ms preprocess, 13.1ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 2
bus 进入:


当前统计:
总进入(下到上): 15 | 总离开(上到下): 23
bus 进入: 1 | 离开: 1
car 进入: 14 | 离开: 21
truck 进入: 0 | 离开: 1
